# 01_1 — Coverage Matrix\n\nสร้าง CSV ตรวจสอบว่าภาพไหนมี/ขาด ครอบคลุม D0–D8, group A/B, condition E/M, view top/side, รหัส 01–30

In [1]:
import os
import pandas as pd
from itertools import product

RAW_DIR = 'raw'

# --- 1. อ่านไฟล์ที่มีจริง เก็บเป็น set ของ (class, id, day, group, cond, view) ---
existing = set()

for class_name in ['cos', 'gok']:
    class_dir = os.path.join(RAW_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    for fname in os.listdir(class_dir):
        if not fname.lower().endswith('.jpg'):
            continue
        name = fname.replace('.jpg', '').replace('.JPG', '').upper()
        prefix = 'COS' if class_name == 'cos' else 'GOK'
        # ตัด prefix ออก แล้วแยกด้วย _ → [id, group, day, cond, view]
        parts = name[len(prefix):].split('_')
        if len(parts) != 5:
            continue
        id_, group, day, cond, view = parts
        existing.add((class_name, id_, day, group, cond, view.lower()))

# --- 2. สร้างแถวทีละ (id, day) ---
IDS  = [f'{i:02d}' for i in range(1, 31)]       # 01–30
DAYS = [f'D{d}' for d in range(9)]              # D0–D8

rows = []
for id_, day in product(IDS, DAYS):
    # กรองเฉพาะ record ที่ตรงกับ id+day นี้
    cos_recs = {(g, c, v) for (cl, i, d, g, c, v) in existing if cl == 'cos' and i == id_ and d == day}
    gok_recs = {(g, c, v) for (cl, i, d, g, c, v) in existing if cl == 'gok' and i == id_ and d == day}
    all_recs = cos_recs | gok_recs

    rows.append({
        'sample': f'{id_}_{day}',
        'cos':    1 if cos_recs else 0,
        'gok':    1 if gok_recs else 0,
        'A':      1 if any(g == 'A' for g, c, v in all_recs) else 0,
        'B':      1 if any(g == 'B' for g, c, v in all_recs) else 0,
        'E':      1 if any(c == 'E' for g, c, v in all_recs) else 0,
        'M':      1 if any(c == 'M' for g, c, v in all_recs) else 0,
        'side':   1 if any(v == 'side' for g, c, v in all_recs) else 0,
        'top':    1 if any(v == 'top'  for g, c, v in all_recs) else 0,
        'temp':   '',
        'note':   '',
    })

df = pd.DataFrame(rows)

print(f'แถวทั้งหมด: {len(df)}  (30 รหัส × 9 วัน)')
print(f'cos มีข้อมูล: {df["cos"].sum()} แถว')
print(f'gok มีข้อมูล: {df["gok"].sum()} แถว')
df.head(18)

แถวทั้งหมด: 270  (30 รหัส × 9 วัน)
cos มีข้อมูล: 230 แถว
gok มีข้อมูล: 220 แถว


,sample,cos,gok,A,B,E,M,side,top,temp,note
0,01_D0,1,1,1,1,1,0,1,1,,
1,01_D1,1,1,1,1,1,1,1,1,,
2,01_D2,1,1,1,1,1,1,1,1,,
3,01_D3,1,1,1,1,0,1,1,1,,
4,01_D4,1,1,1,1,1,1,1,1,,
5,01_D5,1,1,1,1,1,1,1,1,,
6,01_D6,1,1,1,1,1,1,1,1,,
7,01_D7,1,1,1,1,1,1,1,1,,
8,01_D8,1,0,1,1,0,1,1,1,,
9,02_D0,1,1,1,1,1,0,1,1,,


In [2]:
df.to_csv('coverage.csv', index=False)
print(f'บันทึกแล้ว: coverage.csv ({len(df)} แถว)')

บันทึกแล้ว: coverage.csv (270 แถว)
